# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ruchi74/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*


The feature vector contains numerical and categorical information available for analyzing content performance. Numerical missing values are filled using the median, while categorical missing values are filled with `"Unknown"`. Categorical variables are converted into numerical columns using one-hot encoding. Identifiers, outcome-related fields, and potentially future information are excluded to reduce leakage risk. The feature vector is intended for analysis and decision support, not to prove causation.

In [5]:

import pandas as pd
import numpy as np

# Load the anonymized dataset
url = "https://raw.githubusercontent.com/Ruchi74/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

print("Dataset shape:", df.shape)

# Candidate features available in the dataset
numeric_candidates = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "days_with_impressions",
    "days_with_sessions"
]

categorical_candidates = [
    "content_type",
    "main_intent",
    "provider_used",
    "model_used",
    "age_tier",
    "freshness_tier",
    "word_count_tier"
]

numeric_features = [c for c in numeric_candidates if c in df.columns]
categorical_features = [c for c in categorical_candidates if c in df.columns]

# Build feature table
X_raw = df[numeric_features + categorical_features].copy()

# Convert numeric columns and fill missing values with median
for col in numeric_features:
    X_raw[col] = pd.to_numeric(X_raw[col], errors="coerce")
    X_raw[col] = X_raw[col].fillna(X_raw[col].median())

# Fill missing categorical values
for col in categorical_features:
    X_raw[col] = X_raw[col].fillna("Unknown").astype(str)

# One-hot encode categorical columns
X = pd.get_dummies(
    X_raw,
    columns=categorical_features,
    drop_first=False,
    dtype=int
)

print("Original feature columns:", len(X_raw.columns))
print("Encoded feature vector shape:", X.shape)
print("Missing values after preprocessing:", int(X.isna().sum().sum()))
display(X.head())


Dataset shape: (30000, 44)
Original feature columns: 19
Encoded feature vector shape: (30000, 42)
Missing values after preprocessing: 0


,search_volume,competition,cpc,word_count,char_count,content_age_days,days_since_last_update,impressions_90d,clicks_90d,sessions_90d,...,age_tier_91-180,freshness_tier_0-30,freshness_tier_181+,freshness_tier_31-90,freshness_tier_91-180,word_count_tier_1000-2000,word_count_tier_2000-3500,word_count_tier_3500+,word_count_tier_<1000,word_count_tier_Unknown
0,10.0,0.67,2.05,3221.0,20457.0,187,20,3803,29,17,...,0,1,0,0,0,0,1,0,0,0
1,90.0,0.01,0.05,2481.0,15562.0,445,25,15320,7,9,...,0,1,0,0,0,0,1,0,0,0
2,0.0,0.00,0.00,3515.0,23643.0,141,20,12581,11,11,...,1,1,0,0,0,0,0,1,0,0
3,10.0,0.00,0.00,2877.0,19116.0,463,22,11751,58,78,...,0,1,0,0,0,0,0,0,0,1
4,0.0,0.00,0.00,2803.0,17469.0,263,14,19140,24,145,...,0,1,0,0,0,0,1,0,0,0


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*


Numerical features include search volume, competition, cost per click (CPC), word count, character count, content age, days since the last update, impressions, clicks, and sessions. Missing numerical values are filled with the column median.

Categorical features include content type, search intent, provider, model, age tier, freshness tier, and word-count tier. Missing categorical values are replaced with `"Unknown"`, and one-hot encoding converts categories into numeric columns.

Feature availability depends on the prediction moment. Content metadata may be available before a prediction, but traffic measurements collected over a 90-day window may include future information relative to an earlier prediction date. Provider and model fields must also be checked for availability at that time. Therefore, the current feature vector is exploratory until the timing of every feature is verified.

In [7]:

# Section 2: Feature meanings, missing values, and availability

feature_notes = [
    ("search_volume", "Estimated search demand", "Median", "Verify when the search-volume snapshot was collected"),
    ("competition", "Keyword competition score", "Median", "Verify snapshot date"),
    ("cpc", "Cost per click estimate", "Median", "Verify snapshot date"),
    ("word_count", "Number of words in content", "Median", "Available if measured before prediction"),
    ("char_count", "Number of characters in content", "Median", "Available if measured before prediction"),
    ("content_age_days", "Age of content in days", "Median", "Calculate using the prediction date"),
    ("days_since_last_update", "Days since last content update", "Median", "Calculate using the prediction date"),
    ("impressions_90d", "Impressions during a 90-day window", "Median", "Window timing must be verified"),
    ("clicks_90d", "Clicks during a 90-day window", "Median", "Window timing must be verified"),
    ("sessions_90d", "Sessions during a 90-day window", "Median", "Window timing must be verified"),
    ("days_with_impressions", "Days with recorded impressions", "Median", "Verify measurement window"),
    ("days_with_sessions", "Days with recorded sessions", "Median", "Verify measurement window"),
    ("content_type", "Type of content", "Unknown", "Available if recorded before prediction"),
    ("main_intent", "Primary search intent category", "Unknown", "Available if assigned before prediction"),
    ("provider_used", "Content provider used", "Unknown", "Verify whether known at prediction time"),
    ("model_used", "Model used to produce content", "Unknown", "Verify whether known at prediction time"),
    ("age_tier", "Category of content age", "Unknown", "Derive from age at prediction time"),
    ("freshness_tier", "Category based on freshness", "Unknown", "Derive from last update at prediction time"),
    ("word_count_tier", "Category based on word count", "Unknown", "Derive from content available at prediction time")
]

notes_df = pd.DataFrame(
    feature_notes,
    columns=["feature", "meaning", "missing_value_handling", "available_when"]
)

notes_df = notes_df[notes_df["feature"].isin(df.columns)]

print("Number of documented features:", len(notes_df))
display(notes_df)


Number of documented features: 19


,feature,meaning,missing_value_handling,available_when
0,search_volume,Estimated search demand,Median,Verify when the search-volume snapshot was col...
1,competition,Keyword competition score,Median,Verify snapshot date
2,cpc,Cost per click estimate,Median,Verify snapshot date
3,word_count,Number of words in content,Median,Available if measured before prediction
4,char_count,Number of characters in content,Median,Available if measured before prediction
5,content_age_days,Age of content in days,Median,Calculate using the prediction date
6,days_since_last_update,Days since last content update,Median,Calculate using the prediction date
7,impressions_90d,Impressions during a 90-day window,Median,Window timing must be verified
8,clicks_90d,Clicks during a 90-day window,Median,Window timing must be verified
9,sessions_90d,Sessions during a 90-day window,Median,Window timing must be verified


In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*


Feature leakage occurs when a model uses information that would not be available at the moment a prediction is made, or uses information derived from the target. This audit checks for outcome-related columns, future-window measurements, identifiers, and product-generated flags. The dataset includes 90-day traffic measurements and trend fields, so their timing must be checked carefully. The checks identify potential risks; they do not prove that every flagged column is leaking without a defined prediction date and target.

In [9]:

# Section 3: Leakage and privacy checks

# Columns that may reveal outcomes or use future information
suspicious_terms = [
    "trend", "target", "label", "outcome", "health_score",
    "priority_score", "action_type", "quick_win", "needs_ctr_fix",
    "future", "next_30d", "next_90d"
]

suspicious_columns = [
    col for col in df.columns
    if any(term in col.lower() for term in suspicious_terms)
]

# Identifiers should not normally be model features
identifier_columns = [
    col for col in df.columns
    if col.lower() in ["content_id", "client_id", "url", "page_url"]
]

# Review fields that may be measured over a time window
window_columns = [
    col for col in df.columns
    if any(term in col.lower() for term in [
        "90d", "last_30d", "prev_30d", "trend_pct", "trend_direction"
    ])
]

print("Potential outcome/product/future-related columns:")
print(suspicious_columns)

print("\nIdentifier columns to exclude from model features:")
print(identifier_columns)

print("\nTime-window columns requiring timing verification:")
print(window_columns)

# Confirm whether common product flags are present in this dataset
product_flags = [
    "health_score", "needs_ctr_fix", "is_quick_win",
    "priority_score", "action_type"
]

print("\nProduct flags present in dataset:")
print([col for col in product_flags if col in df.columns])

print("\nPrivacy check: raw URL or page URL columns present?")
print([col for col in df.columns
       if "url" in col.lower() or "query" in col.lower()])


Potential outcome/product/future-related columns:
['trend_direction', 'trend_pct']

Identifier columns to exclude from model features:
['content_id', 'client_id']

Time-window columns requiring timing verification:
['impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'trend_direction', 'trend_pct']

Product flags present in dataset:
[]

Privacy check: raw URL or page URL columns present?
[]


In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

### Leakage Hunt: Findings and Verdict

The audit identified `trend_direction` and `trend_pct` as outcome-related fields that should be excluded when predicting future content performance. The identifiers `content_id` and `client_id` are also excluded from model features because they identify records or clients rather than provide generalizable content signals.

The 30-day and 90-day traffic measurements require careful timing checks. They may cause leakage if their measurement windows extend beyond the prediction date. The checked product flags, including `health_score`, `needs_ctr_fix`, `is_quick_win`, `priority_score`, and `action_type`, were not present in this dataset. No URL or query columns were found in the column-name check.

**Verdict:** Potential leakage risks were identified. A definitive leakage conclusion requires a clearly specified prediction date and target. These checks support safer feature selection but do not independently prove that every time-window feature leaks information.

In [11]:

# Section 4: Fields excluded from the feature vector and reasons

excluded_fields = [
    ("content_id", "Record identifier; can encourage memorization."),
    ("client_id", "Client identifier; may reveal client-specific patterns."),
    ("trend_direction", "Potential target-related outcome; exclude when predicting future trends."),
    ("trend_pct", "Contains measured trend information and may reveal the outcome."),
    ("impressions_90d", "Exclude if the prediction happens before this 90-day window ends."),
    ("clicks_90d", "Exclude if clicks are measured after the prediction moment."),
    ("sessions_90d", "Exclude if sessions are measured after the prediction moment."),
    ("pageviews_90d", "Exclude if pageviews are measured after the prediction moment."),
    ("users_90d", "Exclude if user activity is measured after the prediction moment."),
    ("engaged_sessions_90d", "Exclude if engagement is measured after the prediction moment."),
    ("ai_sessions_90d", "Exclude if AI traffic is measured after the prediction moment."),
    ("scroll_events_90d", "Exclude if scroll activity is measured after the prediction moment."),
    ("impressions_last_30d", "Verify the window ends before prediction."),
    ("clicks_last_30d", "Verify the window ends before prediction."),
    ("sessions_last_30d", "Verify the window ends before prediction."),
    ("impressions_prev_30d", "Verify the historical window is available at prediction time."),
    ("clicks_prev_30d", "Verify the historical window is available at prediction time."),
    ("sessions_prev_30d", "Verify the historical window is available at prediction time.")
]

excluded_df = pd.DataFrame(
    excluded_fields,
    columns=["field", "reason_for_exclusion"]
)

excluded_df = excluded_df[excluded_df["field"].isin(df.columns)]

print("Fields documented for exclusion:", len(excluded_df))
display(excluded_df)


Fields documented for exclusion: 18


,field,reason_for_exclusion
0,content_id,Record identifier; can encourage memorization.
1,client_id,Client identifier; may reveal client-specific ...
2,trend_direction,Potential target-related outcome; exclude when...
3,trend_pct,Contains measured trend information and may re...
4,impressions_90d,Exclude if the prediction happens before this ...
5,clicks_90d,Exclude if clicks are measured after the predi...
6,sessions_90d,Exclude if sessions are measured after the pre...
7,pageviews_90d,Exclude if pageviews are measured after the pr...
8,users_90d,Exclude if user activity is measured after the...
9,engaged_sessions_90d,Exclude if engagement is measured after the pr...


### Final Conclusion

The feature vector was built using numerical and categorical features, with median imputation for numerical values and `"Unknown"` for missing categorical values. One-hot encoding produced 42 numerical feature columns with no remaining missing values.

The leakage audit identified record and client identifiers, trend-related fields, and traffic measurements that require exclusion or timing verification. Product flags and URL/query columns were not found in the checked dataset. The feature vector is suitable for exploratory analysis, but it should only be used for prediction after defining the prediction moment and verifying that every feature is available at that time.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.